# Climate Events and Firm Performance — Event Study

## Research question

Do climate events that affect a firm's locations have a measurable impact on its financial and market performance? How persistent are these effects over time?

We study this using a panel of U.S. publicly listed firms matched to county-level climate disaster data (SHELDUS) and annual financial data (Compustat). A firm is "treated" in the year its headquarters county experiences a climate event above a severity threshold. The benchmark paper is Hsu, Lee, Peng, and Yi (2018, *Review of Economics and Statistics*), who show that natural disasters reduce operating performance and that technologically diversified firms are more resilient.

## Empirical strategy

We estimate a staggered difference-in-differences using three complementary estimators:

1. **Sun and Abraham (2021)** interaction-weighted estimator — our baseline.
2. **Stacked DiD** following Cengiz, Dube, Lindner, and Zipperer (2019) — robustness using clean control groups.
3. **de Chaisemartin and D'Haultfœuille (2024)** `did_multiplegt_dyn` — robustness allowing unrestricted treatment effect heterogeneity.

### Sun and Abraham (2021)

> Sun, L., and Abraham, S. (2021). "Estimating Dynamic Treatment Effects in Event Studies with Heterogeneous Treatment Effects." *Journal of Econometrics*, 225(2): 175–199.

The interaction-weighted (IW) estimator corrects the bias of canonical two-way fixed effects (TWFE) when treatment timing varies and effects are heterogeneous across cohorts. The estimating equation is:

$$Y_{it} = \alpha_i + \lambda_t + \sum_{k \neq -1} \beta_k \left( \sum_{g} \mathbb{1}\{G_i = g\} \cdot \mathbb{1}\{t - g = k\} \right) + \varepsilon_{it}$$

where $Y_{it}$ is the outcome for firm $i$ in year $t$, $\alpha_i$ and $\lambda_t$ are firm and year fixed effects, $G_i$ is the cohort (year of first treatment), and $k$ indexes event time. The reference period is $k = -1$. Never-treated firms serve as the control group (coded $G_i = -1000$). The IW-ATT aggregates cohort-specific effects weighted by each cohort's share of treated observations.

Implemented via `fixest::sunab()` in R.

### Stacked DiD (Cengiz et al., 2019)

> Cengiz, D., Dube, A., Lindner, A., and Zipperer, B. (2019). "The Effect of Minimum Wages on Low-Wage Jobs." *Quarterly Journal of Economics*, 134(3): 1405–1454.

The stacked approach constructs cohort-specific sub-experiments, each with its own clean control group of not-yet-treated and never-treated firms, and estimates a pooled regression with cohort-specific fixed effects. This avoids the "forbidden comparisons" (already-treated units serving as controls) that bias TWFE.

### de Chaisemartin and D'Haultfœuille (2024)

> de Chaisemartin, C., and D'Haultfœuille, X. (2024). "Difference-in-Differences Estimators of Intertemporal Treatment Effects." *Review of Economics and Statistics*, forthcoming.

This estimator accommodates unrestricted treatment effect heterogeneity across groups and over time. It identifies "switchers" — firms whose treatment status changes — and computes effects at each post-switch horizon relative to not-yet-switchers and never-switchers. Placebo estimates at pre-switch horizons test parallel trends and no-anticipation.

## Event window

The analysis uses an event window of $k \in [-3, +5]$. Pre-trend coefficients at $k \in \{-3, -2\}$ test parallel trends; post-treatment coefficients $k \in \{0, \ldots, 5\}$ trace the dynamic effect on firm performance.

## Outcome variables

| Variable | Type | Description |
|---|---|---|
| `log_revenue` | Continuous | Log of total revenue (Compustat `revt`) |
| `log_assets` | Continuous | Log of total assets (Compustat `at`) |
| `ln_market_value` | Continuous | Log of market capitalization (price × shares outstanding) |
| `cash_ratio` | Ratio [0,1] | Cash and short-term investments / total assets |

All outcomes are measured at the firm-year level. Log transformations reduce the influence of outliers and allow coefficients to be interpreted as approximate percentage changes.

In [1]:
options(warn = -1)
library(tidyverse)
library(fixest)
library(ggplot2)
library(marginaleffects)
library(did)
library(sandwich)
library(etwfe)
library(arrow)

source("analysis/event_study/r/helpers.R")

-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.5.1
v ggplot2   3.5.2     v tibble    3.3.0
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.1.0     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
  It appears that you are running R and Arrow in emulation (i.e. you're
  running an Intel version of R on a non-Intel mac). This configuration is
  not supported by arrow, you should install a native (arm64) build of R
  and use arrow with that. See https://cran.r-project.org/bin/macosx/



Attachement du package : 'arrow'


L'objet suivant est masqu'e depuis 'package:lubridate':

    duration


L'objet suivant est masqu'e depuis 'package:utils':

    timestamp




In [2]:
DATA_DIR <- "data"

## Data

We load two panel datasets, both constructed from Compustat financials matched to SHELDUS climate events via firm headquarters location (extracted from SEC 10-K filings).

- **Sun panel** (`example_did_staggered.parquet`): Standard firm-year panel. Never-treated firms are coded as cohort $G_i = -1000$. Used for Sun and Abraham (2021) estimation.
- **Roth/stacked panel** (`example_did_stacked.parquet`): Stacked dataset with cohort-specific sub-experiments following Cengiz et al. (2019). Each treated cohort is paired with its own clean control group of not-yet-treated and never-treated firms.
- **Multi-event panel** (`example_did_multi_event.parquet`): Panel with binary absorbing treatment $D_{it} = \mathbb{1}\{t \geq g_i\}$. Used for de Chaisemartin and D'Haultfœuille (2024).

Treatment is defined at the firm level: a firm is treated in the first year its headquarters county experiences a climate event above the severity threshold.

In [3]:
df_raw_sun <- read_parquet(
  file.path(DATA_DIR, "example_did_staggered.parquet")
)
sun <- prepare_sunab(df_raw_sun)

df_final            <- sun$df_full
dict_pretty <- make_pretty_dict(min_k = -3, max_k = 5)

dim(df_final)

[1] 21567    89

## 1. Baseline: Sun and Abraham (2021) — OLS

We estimate the Sun and Abraham (2021) interaction-weighted estimator via `fixest::feols()` with `sunab()`, using firm and year fixed effects and standard errors clustered at the firm level.

For each outcome, we report both:
- **ATT** (aggregate treatment effect on the treated), and
- **Event-study** coefficients $\hat{\beta}_k$ for $k \in \{-3, \ldots, +5\}$, with $k = -1$ as reference.

| Column | Outcome | Interpretation |
|---|---|---|
| (1)–(2) | `log_revenue` | Revenue effect (≈ % change) |
| (3)–(4) | `log_assets` | Asset base effect (≈ % change) |
| (5)–(6) | `ln_market_value` | Market valuation effect (≈ % change) |
| (7)–(8) | `cash_ratio` | Liquidity effect (level change in cash/assets) |

Pre-trend coefficients at $k \in \{-3, -2\}$ should be close to zero and statistically insignificant if the parallel trends assumption holds.

In [4]:
t0 <- make_feols_sunab(
  df = df_final , # %>% filter(has_variation_climate_risk == 1)
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "log_revenue"
)

t1 <- make_feols_sunab(
  df = df_final,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "log_revenue"
)

t2 <- make_feols_sunab(
  df = df_final,
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "log_assets"
)

t3 <- make_feols_sunab(
  df = df_final,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "log_assets"
)

t4 <- make_feols_sunab(
  df = df_final,
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "ln_market_value"
)

t5 <- make_feols_sunab(
  df = df_final,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "ln_market_value"
)

t6 <- make_feols_sunab(
  df = df_final,
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "cash_ratio"
)

t7 <- make_feols_sunab(
  df = df_final,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "cash_ratio"
)

etable(
  t0,
  t1,
  t2,
  t3,
  t4,
  t5,
  t6,
  t7,
  cluster = "cik", style.df = my_style, drop = " x cohort = ",
  dict = dict_pretty, digits = 3
)

NOTE: 6,232 observations removed because of NA values (LHS: 6,232).

NOTE: 6,232 observations removed because of NA values (LHS: 6,232).

NOTE: 6,181 observations removed because of NA values (LHS: 6,181).

NOTE: 6,181 observations removed because of NA values (LHS: 6,181).

NOTE: 7,416 observations removed because of NA values (LHS: 7,416).

NOTE: 7,416 observations removed because of NA values (LHS: 7,416).

NOTE: 8,913 observations removed because of NA values (LHS: 8,913).

NOTE: 8,913 observations removed because of NA values (LHS: 8,913).



,,t0,t1,t2,t3,t4,t5,t6,t7
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Dependent Var.:,log_revenue,log_revenue,log_assets,log_assets,ln_market_value,ln_market_value,cash_ratio,cash_ratio
2,,,,,,,,,
3,Treated x Post,0.054 (0.034),,0.024 (0.029),,-0.021 (0.037),,0.006 (0.007),
4,Treated x t-3,,-0.003 (0.043),,-0.034 (0.036),,-0.082 (0.052),,-0.002 (0.010)
5,Treated x t-2,,0.021 (0.029),,-0.017 (0.026),,-0.070* (0.038),,-0.004 (0.007)
6,Treated x t+0,,0.060*** (0.019),,0.026* (0.015),,0.012 (0.024),,0.002 (0.005)
7,Treated x t+1,,0.058* (0.030),,0.044** (0.022),,-0.001 (0.034),,0.003 (0.006)
8,Treated x t+2,,0.045 (0.036),,0.032 (0.030),,-0.039 (0.042),,0.008 (0.007)
9,Treated x t+3,,0.056 (0.043),,0.016 (0.038),,-0.038 (0.048),,0.006 (0.009)


## 2. Robustness: Stacked DiD (Cengiz et al., 2019)

As a first robustness check, we re-estimate the same four outcomes using a stacked difference-in-differences design following Cengiz, Dube, Lindner, and Zipperer (2019). Each treated cohort is matched to a clean control group of not-yet-treated and never-treated firms, avoiding "forbidden comparisons" where already-treated firms serve as controls.

The regression includes cohort-specific group and time fixed effects (`id_group + group_year`) and is estimated via `fixest::feols()` with standard errors clustered at the firm level. This approach is more conservative than Sun and Abraham (2021) because it restricts the comparison set, but it is robust to a wider class of treatment effect heterogeneity.

In [9]:
df_raw_roth <- read_parquet(
  file.path(DATA_DIR, "example_did_stacked.parquet")
)
df_roth <- prepare_roth(df_raw_roth)$df_full

In [11]:
t0 <- make_feols_roth(
  df = df_roth , # %>% filter(has_variation_climate_risk == 1)
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "log_revenue"
)

t1 <- make_feols_roth(
  df = df_roth,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "log_revenue"
)

t2 <- make_feols_roth(
  df = df_roth,
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "log_assets"
)

t3 <- make_feols_roth(
  df = df_roth,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "log_assets"
)

t4 <- make_feols_roth(
  df = df_roth,
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "ln_market_value"
)

t5 <- make_feols_roth(
  df = df_roth,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "ln_market_value"
)

t6 <- make_feols_roth(
  df = df_roth,
  covariates = NULL,
  event_study = FALSE,
  cluster = "cik",
  outcome_var = "cash_ratio"
)

t7 <- make_feols_roth(
  df = df_roth,
  covariates = NULL,
  event_study = TRUE,
  cluster = "cik",
  outcome_var = "cash_ratio"
)

etable(
  t0,
  t1,
  t2,
  t3,
  t4,
  t5,
  t6,
  t7,
  cluster = "cik", style.df = my_style, drop = " x cohort = ",
  dict = dict_pretty, digits = 3
)

NOTE: 24,800 observations removed because of NA values (LHS: 24,800).

NOTE: 24,800 observations removed because of NA values (LHS: 24,800).

NOTE: 24,645 observations removed because of NA values (LHS: 24,645).

NOTE: 24,645 observations removed because of NA values (LHS: 24,645).

NOTE: 29,337 observations removed because of NA values (LHS: 29,337).

NOTE: 29,337 observations removed because of NA values (LHS: 29,337).

NOTE: 36,035 observations removed because of NA values (LHS: 36,035).

NOTE: 36,035 observations removed because of NA values (LHS: 36,035).



,,t0,t1,t2,t3,t4,t5,t6,t7
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Dependent Var.:,log_revenue,log_revenue,log_assets,log_assets,ln_market_value,ln_market_value,cash_ratio,cash_ratio
2,,,,,,,,,
3,treated x post,0.081*** (0.027),,0.085*** (0.023),,-0.012 (0.033),,-0.004 (0.005),
4,treated x event_time = -3,,-0.041 (0.040),,-0.068** (0.032),,-0.141*** (0.046),,-0.008 (0.009)
5,treated x event_time = -2,,-0.010 (0.029),,-0.030 (0.026),,-0.100*** (0.035),,0.0005 (0.007)
6,treated x event_time = 0,,0.062*** (0.021),,0.043** (0.018),,-0.046* (0.026),,0.0003 (0.006)
7,treated x event_time = 1,,0.065** (0.029),,0.065*** (0.022),,-0.119*** (0.034),,-0.009 (0.006)
8,treated x event_time = 2,,0.072** (0.033),,0.082*** (0.027),,-0.095** (0.041),,-0.009 (0.008)
9,treated x event_time = 3,,0.081** (0.038),,0.057* (0.034),,-0.045 (0.046),,-0.006 (0.008)


## 3. Robustness: de Chaisemartin and D'Haultfœuille (2024) estimator

> de Chaisemartin, C., and D'Haultfœuille, X. (2024). "Difference-in-Differences Estimators of Intertemporal Treatment Effects." *Review of Economics and Statistics*, forthcoming.

As a second robustness check, we re-estimate using the `did_multiplegt_dyn` estimator. Unlike Sun and Abraham (2021), this estimator allows unrestricted treatment effect heterogeneity across groups and over time without imposing functional-form restrictions.

The estimator identifies "switchers" — firms whose treatment status changes from $D = 0$ to $D = 1$ — and computes effects at each post-switch horizon relative to not-yet-switchers and never-switchers. Placebo estimates at pre-switch horizons test the parallel trends and no-anticipation assumptions.

The treatment variable $D_{it} = \mathbb{1}\{t \geq g_i\}$ is a binary absorbing indicator. Effects are estimated at 3 post-treatment horizons with 3 pre-treatment placebos.

In [12]:
library(DIDmultiplegtDYN)
library(data.table)
library(plm)
library(haven)
library(MASS)
library(matlib)
library(cowplot)

pkg_path <- "analysis/event_study/r/dechaisemartin/dechaisemartin"
for (f in list.files(pkg_path, pattern = "\\.R$", full.names = TRUE)) source(f)


Attachement du package : 'data.table'


Les objets suivants sont masqu'es depuis 'package:lubridate':

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqu'es depuis 'package:dplyr':

    between, first, last


L'objet suivant est masqu'e depuis 'package:purrr':

    transpose



Attachement du package : 'plm'


L'objet suivant est masqu'e depuis 'package:data.table':

    between


Les objets suivants sont masqu'es depuis 'package:dplyr':

    between, lag, lead



Attachement du package : 'MASS'


L'objet suivant est masqu'e depuis 'package:dplyr':

    select



Attachement du package : 'matlib'


L'objet suivant est masqu'e depuis 'package:fixest':

    ref



Attachement du package : 'cowplot'


L'objet suivant est masqu'e depuis 'package:lubridate':

    stamp




In [13]:
# Load multi-event panel (D column built by did-panel-builder)
df_me <- read_parquet(
  file.path(DATA_DIR, "example_did_multi_event.parquet")
) %>%
  distinct(cik, year, .keep_all = TRUE) %>%
  mutate(
    cik_num = as.numeric(cik)
  )


In [20]:
specs <- list(
  list(label = "log(Revenue)",     outcome = "log_revenue"),
  list(label = "log(Assets)",      outcome = "log_assets"),
  list(label = "ln(Market Value)", outcome = "ln_market_value"),
  list(label = "Cash Ratio",       outcome = "cash_ratio")
)

In [21]:
# Baseline: 5 outcomes x (3 effects + 3 placebos)
results <- run_dcdh_specs(df_me,  effects = 3, placebo = 3, specs = specs)
build_dcdh_table(results, n_obs = nrow(df_me))

  log(Revenue) ... 

Placebo_3cannot be estimated. There is no switcher or no control for this placebo.

Some placebos could not be estimated. Therefore, the test of joint nullity of the placebos could not be computed.



ATT = -0.0069 (SE = 0.0822)
  log(Assets) ... 

Placebo_3cannot be estimated. There is no switcher or no control for this placebo.

Some placebos could not be estimated. Therefore, the test of joint nullity of the placebos could not be computed.



ATT = 0.0341 (SE = 0.0769)
  ln(Market Value) ... 

Placebo_3cannot be estimated. There is no switcher or no control for this placebo.

Some placebos could not be estimated. Therefore, the test of joint nullity of the placebos could not be computed.



ATT = 0.0057 (SE = 0.0967)
  Cash Ratio ... ATT = -0.0014 (SE = 0.0141)


,log(Revenue) ATT,log(Revenue) ES,log(Assets) ATT,log(Assets) ES,ln(Market Value) ATT,ln(Market Value) ES,Cash Ratio ATT,Cash Ratio ES
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Placebo 3,,NANA,,NANA,,NANA,,-0.0028
Placebo 2,,0.0041,,0.0659*,,0.0294,,-0.0091
Placebo 1,,0.0133,,0.0542,,0.1970**,,0.0235
Effect 1,,0.0081,,0.0230,,-0.1316,,-0.0092
Effect 2,,-0.0339,,-0.0054,,0.1495,,0.0086
Effect 3,,0.0136,,0.0959,,0.1563*,,0.0031
ATT,-0.0069,-0.0069,0.0341,0.0341,0.0057,0.0057,-0.0014,-0.0014
,(0.0822),(0.0822),(0.0769),(0.0769),(0.0967),(0.0967),(0.0141),(0.0141)
N,"27,889","27,889","27,889","27,889","27,889","27,889","27,889","27,889"


### Interpreting the dCDH results

The table reports ATT and event-study estimates for each outcome. Each outcome has two columns: the aggregate ATT and the full event-study path (3 placebos + 3 effects).

- **Placebo rows** (Placebo 1–3): pre-treatment coefficients testing parallel trends. Under the null, these should be zero.
- **Effect rows** (Effect 1–3): post-treatment dynamic effects at horizons $k = +1, +2, +3$.
- **ATT**: weighted average of all post-treatment effects.
- **Joint placebo p**: Wald test of $H_0$: all placebo coefficients are jointly zero. This is the formal parallel trends test.
- **Joint effects p**: Wald test of $H_0$: all post-treatment effects are jointly zero. Tests whether the treatment had any detectable impact across horizons.

### Note on pre-trend composition

If individual placebo coefficients appear significant while the joint test does not reject, this may reflect compositional shifts in the pre-treatment window rather than a true pre-trend violation. When a large share of switchers contribute only one pre-treatment observation, distant placebos are identified from a selected subset of later-cohort firms. Restriction A (below), which requires at least 3 pre-treatment periods per switcher, eliminates this compositional artifact.